In [ ]:
# import sys
# # !bash -c "source ~/.bashrc"
# sys.path.append('/home/franckd/ros2_ws/install/lib/python3.10/site-packages')  # Adjust Python version

In [ ]:
import os
from bng_simulator.utils.services_utils import send_request
from bng_simulator.utils.io_dict_utils import(
    round_dict_values,
    build_tree_from_dict,
    convert_tree_into_proper_dict
)

In [ ]:
# Let's check the teleport functionality
teleport_args = {
    "yaw_angle": -90,
}
send_request("teleport_vehicle", teleport_args)

In [ ]:
# Let's check the ABS functionality
send_request("get_ABS")

In [ ]:
# Set the ABS
abs_args = {
    "enabled": False
}
send_request("set_ABS", abs_args)

In [ ]:
# Get ESC status
send_request("get_ESC")

In [ ]:
# Set the ESC to False,
esc_args = {
    "enabled": True
}
send_request("set_ESC", esc_args)

In [ ]:
# Get the 4WD status
send_request("get_4WD_mode")

In [ ]:
# Set the 4WD to True
_4wd_args = {
    "mode": "2WD",
    "range_mode": "high" # May not matter for 2WD
}
send_request("set_4WD_mode", _4wd_args)

In [ ]:
# Get lock diffs status
lock_diffs_args = {
    "diff" : "front"
}
send_request("get_diff_lock_state", lock_diffs_args)

In [ ]:
# Set lock diffs
lock_diffs_args = {
    "diff" : "front",
    "lock" : True
}
send_request("lock_diff", lock_diffs_args)

In [ ]:
# Part config of current vehicle
vehicle_part = send_request("get_vehicle_part_config")
# partConfigFilename is relevant for unique vehicle conf.
vehicle_main_file = vehicle_part["partConfigFilename"]

In [ ]:
# Get gear information
send_request("get_gearbox_info")

In [ ]:
import time

# Let's enumerate all gear options and their characteristics
def extract_gear_infos(vehicle_name = None):
    """
    Go through all gears and extract their characteristics
    """
    in_args = {}
    in_args_set = {}
    if vehicle_name is not None:
        in_args["vehicle_name"] = vehicle_name
        in_args_set["vehicle_name"] = vehicle_name
    # Let's first get current infos
    gear_infos = send_request("get_gearbox_info", in_args)
    min_gear = int(gear_infos["minGearIndex"])
    max_gear = int(gear_infos["maxGearIndex"])
    # TODO: The min is actually negative of the actual min cause of abs.
    print(f"Min/max gear: {min_gear}/{max_gear}")
    
    # A function to parse the gear infos
    def parse_gear_infos(gear_infos):
        curr_gear = int(gear_infos["gearIndex"])
        gear_mode_index = gear_infos["gearModeIndex"]
        gear_ratio = gear_infos["gearRatio"]
        gear_name = gear_infos["gearName"]
        gear_mode = gear_infos["mode"]
        return gear_name, \
            {"index": curr_gear, "mode_index": gear_mode_index, 
             "gear_ratio": gear_ratio, "mode": gear_mode}
    
    # Store the different gear ratio
    mapping_info = {}
    gear_name, gear_info = parse_gear_infos(gear_infos)
    print(f"Gear {gear_name} - {gear_info}")
    mapping_info[gear_name] = gear_info
    initial_gear_index = int(gear_infos["gearIndex"])
    for i in range(-2, max_gear + 2): # Just extra for the sake of it
        in_args_set["gear_index"] = i
        send_request("set_gearbox_index", in_args_set)
        # Wait for a bit
        time.sleep(0.5)
        # Get the gear infos
        gear_infos = send_request("get_gearbox_info", in_args)
        gear_name, gear_info = parse_gear_infos(gear_infos)
        print(f"Gear {gear_name} - {gear_info}")
        mapping_info[gear_name] = gear_info
    # Set the initial gear index
    in_args_set["gear_index"] = initial_gear_index
    send_request("set_gearbox_index", in_args_set)
    return mapping_info

In [ ]:
# Let's extract the gear infos
extract_gear_infos()

In [ ]:
def format_powertrain_structure(data, relevant_keys):
    """
    Format the powertrain structure.
    
    Args:
        data (Dict[str, Any]): The data.
        relevant_keys (Tuple[str]): The relevant keys.
        
    Returns:
        Dict[str, Any]: The tree-like structure.
    """
    roots, tree = build_tree_from_dict(data)
    res = {}
    for root in roots:
        res[root] = convert_tree_into_proper_dict(root, tree, data, relevant_keys)
    # # Let;s sort it
    # res = dict
    return round_dict_values(res)

In [ ]:
# Powertrain infos
powertrain_prop = send_request("get_powertrain_properties")
formatted_powertrain = format_powertrain_structure(
    powertrain_prop,
    ['type', 'gearRatio', 'gearRatios', 'availableModes', 'diffTorqueSplit']
)
powertrain_prop

In [ ]:
# Get controllers infos
controllers_infos = send_request("get_controller_infos")
controllers_infos = dict(sorted(controllers_infos.items()))
controllers_infos

In [ ]:
import numpy as np

In [ ]:
vDiff = np.array(kin_props["cogPosRelRef"]) - np.array(kin_props["posRef"])
vDiff

In [ ]:
np.array(kin_props["currPos"]) + vDiff

In [ ]:
# Kinematics infos
kin_args = {
    "without_wheels": False,
    "world_space": True
}
kin_props = send_request("get_vehicle_properties", kin_args)
kin_props

In [ ]:
kin_props